In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze = "healthcare_adjudication.healthcare_adjudication_bronze"
silver = "healthcare_adjudication.healthcare_adjudication_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver}")


In [0]:
%sql
ALTER TABLE healthcare_adjudication.healthcare_adjudication_silver.policies ADD COLUMN start_date DATE;
ALTER TABLE healthcare_adjudication.healthcare_adjudication_silver.policies ADD COLUMN end_date DATE;
ALTER TABLE healthcare_adjudication.healthcare_adjudication_silver.policies ADD COLUMN is_current BOOLEAN;

In [0]:
bronze = "healthcare_adjudication.healthcare_adjudication_bronze"
silver = "healthcare_adjudication.healthcare_adjudication_silver"

from pyspark.sql.functions import current_date, lit

# Read current policies table
policies = spark.table(f"{silver}.policies")

# Add SCD2 columns for tracking
policies_scd2 = policies \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

# Overwrite table with new columns
policies_scd2.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{silver}.policies")

print("SCD2 columns added and initialized in policies table.")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit, col

silver_schema = "healthcare_adjudication.healthcare_adjudication_silver"
bronze_schema = "healthcare_adjudication.healthcare_adjudication_bronze"

# Load updated policy records from bronze
source = spark.table(f"{bronze_schema}.policies")

# Transform incoming updates, flattening and aligning columns
source_df = source.select(
    col("policy_id"),
    col("member.patient_id").alias("patient_id"),
    col("payer.payer_id").alias("payer_id"),
    col("coverage.plan").alias("plan"),
    col("coverage.limit").alias("coverage_limit"),
    col("coverage.copay").alias("copay"),
    # SCD2 tracking columns
    lit(current_date()).alias("start_date"),
    lit(None).cast("date").alias("end_date"),
    lit(True).alias("is_current")
)

# Reference Delta table
delta_table = DeltaTable.forName(spark, f"{silver_schema}.policies")

merge_condition = """target.policy_id = source.policy_id AND target.is_current = True AND (
    target.plan <> source.plan OR
    target.coverage_limit <> source.coverage_limit OR
    target.copay <> source.copay OR
    target.patient_id <> source.patient_id OR
    target.payer_id <> source.payer_id
)"""

# SCD2 Update: deactivate old records for updated policy (is_current -> False, end_date -> now)
delta_table.alias("target").merge(
    source_df.alias("source"),
    merge_condition
).whenMatchedUpdate(
    set={
        "is_current": "false",
        "end_date": "current_date()"
    }
).whenNotMatchedInsertAll()

print("SCD2 MERGE completed: policies table updated with history in-place.")

In [0]:
providers = (
    spark.table(f"{bronze}.providers")
    .dropDuplicates(["provider_id"])
)

providers.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.providers")


In [0]:
payers = (
    spark.table(f"{bronze}.payers")
    .dropDuplicates(["payer_id"])
)

payers.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.payers")


In [0]:
patients = (
    spark.table(f"{bronze}.patients")
    .withColumn("dob", to_date("dob"))
    .dropDuplicates(["patient_id"])
)

patients.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.patients")


In [0]:
policies = (
    spark.table(f"{bronze}.policies")
    .select(
        col("policy_id"),
        col("member.patient_id").alias("patient_id"),
        col("payer.payer_id").alias("payer_id"),
        col("coverage.plan").alias("plan"),
        col("coverage.limit").alias("coverage_limit"),
        col("coverage.copay").alias("copay")
    )
)

policies.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.policies")

Claims

In [0]:
claims = (
    spark.table(f"{bronze}.claims")
    .withColumn("claim_date", to_date("claim_date"))
    .dropDuplicates(["claim_id"])
)

claims.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.claims")


In [0]:

claim_lines = (
    spark.table(f"{bronze}.claim_lines")
    .withColumn("procedure", explode("procedures"))
    .select(
        col("claim_id"),
        col("procedure.code").alias("procedure_code"),
        col("procedure.charge").alias("procedure_charge")
    )
)

claim_lines.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.claim_lines")



In [0]:
payments = (
    spark.table(f"{bronze}.payments")
    .dropDuplicates(["payment_id"])
)

payments.write.format("delta").mode("overwrite") \
.saveAsTable(f"{silver}.payments")


In [0]:
%sql
use healthcare_adjudication.healthcare_adjudication_gold

In [0]:
%sql
select * from claim_approval_rate limit10

In [0]:
%sql
use healthcare_adjudication_silver

In [0]:
%sql
select * from providers where provider_id = 2001


In [0]:
%sql
SELECT*
FROM healthcare_adjudication.healthcare_adjudication_bronze.policies
LIMIT 5;


In [0]:
%sql
UPDATE healthcare_adjudication.healthcare_adjudication_bronze.policies
SET coverage = named_struct(
    'plan', coverage.plan,
    'limit', 999999,
    'copay', coverage.copay
)
WHERE policy_id = 1;


In [0]:
%sql
SELECT policy_id, coverage.limit
FROM healthcare_adjudication.healthcare_adjudication_bronze.policies
WHERE policy_id = 1;


In [0]:
%sql
select * from healthcare_adjudication.healthcare_adjudication_silver.policies limit 5

In [0]:
%sql
DROP TABLE IF EXISTS healthcare_adjudication.healthcare_adjudication_silver.policies_scd2;